# COMP 6934 W 26 Note 20

## Choropleth Maps

Munzner's description of the choropleth map idiom from slides 

In [1]:
# Slideviewer code to use Tamara Munzner's book slides
from IPython.display import Image

slideset = [103, 352, 357, 361, 364, 365, 367]

folder="slides2021munzner/"
def Munzner21slide(no):  
    return folder+"slide-{:03d}.jpg".format(no)

slide_no=0
slide_list=[Munzner21slide(v) for v in slideset]

In [2]:
# Show the slide and increment slide_no for the next one.
display(Image(url=slide_list[slide_no]))
slide_no = (slide_no +1) % len(slide_list)

# Choropleth map in plotly

A basic introduction is provided at https://plotly.com/python/choropleth-maps/

In particular, note:
* The alternative technology for geo-map rendering is tiling, which has advantages in zooming scale, details, and responsiveness, but is locked into the pre-generated tile sets.
* Plotly renders to a built-in world map, so all maps are referenced to its co-ordinate system
* Plotly has built in map information for US to the county level, but we have to import our own geo shapes for other maps
* Other packages are better at providing more capabilities for geo-maps, so explore them if you need to. ArcGIS is a well known (but with a paywall) one.
* Basic choropleth maps are available in matplotlib and other packages, nothin special about plotly in particular.
* geoJSON is one format for map information.
    * Some others are discussed in this post: https://medium.com/@limeira.felipe94/which-format-to-use-shapefile-geojson-and-geopackage-198ef9f5e00f
    * https://en.wikipedia.org/wiki/GeoJSON the wikipedia entry is easier to follow than the standard

In [ ]:
import json

In [ ]:
canprov = json.load(open("datasets/canada.geojson"))

In [ ]:
canprov

In [ ]:
for i in canprov['features']:
    i['properties']['prov_name'] =  i['properties']['prov_name_en'][0]
    print(i['properties'])

In [ ]:
import pandas as pd


In [ ]:
can_df=  pd.read_csv("datasets/canadapop.csv", thousands=",")
can_df

In [ ]:
geonames = [i['properties']['prov_name'] for i in canprov['features']]
dfnames = can_df['Geography']
dfnames

In [ ]:
ndf = can_df.drop(index=[0,1])
ndf

In [ ]:
ndf['Geography'][13] = 'Northwest Territories'
ndf['Geography'][14] = 'Nunavut'
ndf.info()

In [ ]:
ndf

In [ ]:
geonames

In [ ]:
!pip install geojson_rewind

In [ ]:
from geojson_rewind import rewind
canprov_corrected=rewind(canprov,rfc7946=False)


In [ ]:
import plotly.express as px

fig2 = px.choropleth(ndf, geojson=canprov_corrected, locations='Geography', color='Q4 2024',
                           featureidkey="properties.prov_name",
                           color_continuous_scale="Viridis_r",
                           range_color=(0, 20000000),
                           # projection="mercator"
                          )
fig2.update_geos(fitbounds="locations", visible=True)
fig2.update_layout(
    margin=dict(l=20, r=20, t=20, b=20),
)
fig2.show()

In [ ]:
print(fig2)

In [ ]:
for i in canprov_corrected['features']:
    print(i['properties']['geo_point_2d'], i['properties']['prov_name'])

In [ ]:
import plotly.graph_objects as go


trace = go.Scattergeo(lon = [i['properties']['geo_point_2d']['lon'] for i in canprov_corrected['features']],
                    lat = [i['properties']['geo_point_2d']['lat'] for i in canprov_corrected['features']],
                    mode="markers")
fig2.add_trace(trace)

In [ ]:
import plotly.express as px

fig3 = px.choropleth(ndf, geojson=canprov_corrected, locations='Geography', color='Q4 2024',
                           featureidkey="properties.prov_name",
                           color_continuous_scale="Viridis_r",
                           range_color=(0, 20000000),
                           # projection="mercator"
                          )
fig3.update_geos(fitbounds="locations", visible=True)

traceSpot = go.Scattergeo(lon = [i['properties']['geo_point_2d']['lon'] for i in canprov_corrected['features']],
                         lat = [i['properties']['geo_point_2d']['lat'] for i in canprov_corrected['features']],
                     mode="text",
                         text=[i['properties']['prov_name'] for i in canprov_corrected['features']],
                         )
fig3.add_trace(traceSpot)
fig3.update_layout(
    margin=dict(l=20, r=20, t=20, b=20),
)

fig3.show()

In [ ]:
import plotly.express as px

fig4 = px.choropleth(ndf, geojson=canprov_corrected, locations='Geography', color='Q4 2024',
                           featureidkey="properties.prov_name",
                           color_continuous_scale="oranges",
                           range_color=(0, 20000000),
                           # projection="mercator"
                          )
fig4.update_geos(fitbounds="locations", visible=True)

traceSpot = go.Scattergeo(lon = [i['properties']['geo_point_2d']['lon'] for i in canprov_corrected['features']],
                         lat = [i['properties']['geo_point_2d']['lat'] for i in canprov_corrected['features']],
                     mode="text",
                         text=[i['properties']['prov_name'] for i in canprov_corrected['features']],
                          textposition="bottom center",
                          
                         )
fig4.add_trace(traceSpot)

fig4.update_layout(
    margin=dict(l=20, r=20, t=20, b=20),
)
fig4.show()

In [ ]:
cancities=pd.read_csv('datasets/canadacities.csv')
cancities

In [ ]:
capitals="""Edmonton
Victoria
Winnipeg
Fredericton
St. John's
Halifax
Toronto
Charlottetown
Quebec City
Regina
Whitehorse
Iqaluit
Yellowknife""".split('\n')

In [ ]:
capitalDF=cancities[cancities['city'].isin(capitals)].reset_index()
capitalDF

In [ ]:
capitalDF.drop([13,14], inplace=True) # extra victorias

In [ ]:
capitalDF.info()

In [ ]:
capitalDF

In [ ]:

traceCapital = go.Scattergeo(lon = capitalDF['lng'],
                             lat = capitalDF['lat'],
                             mode="markers",
                             marker_symbol="star",
                             marker_color="blue",
                              text=capitalDF['city'],
                          
                         )
    

fig4.add_trace(traceCapital)


fig4.show()

# Attributions

Be sure to add your own sources or indicate you have none to add.  Sources can be web sites, text materials, and so on. They do not have to be hyperlinks. Other people are also sources, but they are not allowed for in class credit problems.

| Source | What is it | How used |
|--|--|--|
| https://simplemaps.com/data/canada-cities | data for canadian cities | |
| https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1710000901 | Canadian quarterly population estimates ||
| https://github.com/plotly/plotly.py/blob/main/doc/python/choropleth-maps.md | plotly chloropleth maps | code examples |
| https://data.opendatasoft.com/explore/dataset/georef-canada-province%40public/export/?disjunctive.prov_name_en&flg=en-us | geojson for Canadian provinces |
| https://plotly.com/python/scatter-plots-on-maps/ | plotly scattergeo traces | code examples |
| https://plotly.com/python/reference/scattergeo/#scattergeo | plotly scatter-geo reference | parameter definitions |

